In [1]:
import mortapy as mp
import math
import os
#import traceback

In [2]:
# examples/demo_momen_lengkap.py

# from IPython.display import display, Markdown # Jika ingin menjalankan di Jupyter dengan %run
# Fungsi helper
def tampilkan_hasil_momen(deskripsi_awal: str, hasil_objek: mp.ActuarialResult, verifikasi_manual_str: str = ""): # Ubah nama parameter
    print(deskripsi_awal)
    hasil_objek.show() # Ini akan memanggil _repr_latex_ di Jupyter atau print Deskripsi di terminal
    if verifikasi_manual_str: # Cek parameter yang sudah diubah namanya
        print(verifikasi_manual_str)
    print("-" * 50)

# --- Definisi Parameter Umum ---
print("=" * 70)
print("          DEMO LENGKAP PERHITUNGAN MOMEN MORTAPY")
print("=" * 70)

usia_x = 65
suku_bunga_untuk_kalkulator = 0.05 # Meskipun tidak selalu dipakai untuk momen, kalkulator butuh ini
gender_pilihan_tabel = 'wanita'
n_temporary = 10 # Untuk momen temporary

# Parameter untuk Asumsi
qx_konstan_val = 0.03
omega_dm_val = 100.0
alpha_beta_val = 1.0 # General De Moivre dengan alpha=1 sama dengan De Moivre standar
mu_cfm_val = 0.025
gompertz_params_val = [0.0001, 1.1] # B, c
makeham_params_val = [0.0002, 0.00008, 1.12] # A, B, c

print("\n--- Parameter Umum yang Digunakan dalam Demo ---")
print(f"Usia awal (x)         : {usia_x}")
print(f"Suku bunga (i)        : {suku_bunga_untuk_kalkulator:.2%}")
print(f"Gender (untuk tabel)  : {gender_pilihan_tabel.capitalize()}")
print(f"Periode temporary (n) : {n_temporary} tahun")
print("-" * 50)


# ==============================================================================
# BAGIAN 1: MOMEN CURTATE BERBASIS TABEL MORTALITA (TMI DEFAULT)
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 1: MOMEN CURTATE BERBASIS TABEL MORTALITA (TMI DEFAULT)")
print("=" * 70)

try:
    tabel_default = mp.load_default_table()
    print(f"\nBerhasil memuat tabel default: {tabel_default}\n")
    print(f"--- Menggunakan parameter: Usia = {usia_x}, Gender = {gender_pilihan_tabel}, n_temporary = {n_temporary} ---")

    # --- 1.1 Momen Pertama Curtate (e_x dan e_{x:n|}) ---
    ex_wl_tabel = mp.ex_curtate_table(
        age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel
    )
    tampilkan_hasil_momen(f"1.1.1 Ekspektasi Curtate Future Lifetime (e_{{{usia_x}}}, Whole Life, Tabel):", ex_wl_tabel)
    
    ex_wl_tabel_untuk_ops = ex_wl_tabel

    ex_temp_tabel = mp.ex_curtate_table(
        age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary
    )
    tampilkan_hasil_momen(f"1.1.2 Ekspektasi Curtate Future Lifetime (e_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temporary, Tabel):", ex_temp_tabel)

    # --- 1.2 Momen Kedua Curtate (E[K_x^2] dan E[(K_{x:n|})^2]) ---
    e_sq_wl_tabel = mp.e_sq_curtate_table(
        age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel
    )
    tampilkan_hasil_momen(f"1.2.1 Momen Kedua Curtate (E[K_{{{usia_x}}}^2], Whole Life, Tabel):", e_sq_wl_tabel)

    e_sq_temp_tabel = mp.e_sq_curtate_table(
        age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary
    )
    tampilkan_hasil_momen(f"1.2.2 Momen Kedua Curtate (E[K_{{{usia_x}:\\overline{{{n_temporary}}}|}}^2], Temporary, Tabel):", e_sq_temp_tabel)

    # --- 1.3 Variansi Curtate (Var(K_x) dan Var(K_{x:n|})) ---
    var_k_wl_tabel = mp.var_k_table(
        age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel
    )
    tampilkan_hasil_momen(f"1.3.1 Variansi Curtate (Var[K_{{{usia_x}}}], Whole Life, Tabel):", var_k_wl_tabel,
                         verifikasi_manual_str=f"    -> Verifikasi: {e_sq_wl_tabel.value:.4f} - ({ex_wl_tabel.value:.4f})^2 = {e_sq_wl_tabel.value - ex_wl_tabel.value**2:.4f}")

    var_k_temp_tabel = mp.var_k_table(
        age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, gender=gender_pilihan_tabel, n_temp=n_temporary
    )
    tampilkan_hasil_momen(f"1.3.2 Variansi Curtate (Var[K_{{{usia_x}:\\overline{{{n_temporary}}}|}}], Temporary, Tabel):", var_k_temp_tabel,
                          verifikasi_manual_str=f"    -> Verifikasi: {e_sq_temp_tabel.value:.4f} - ({ex_temp_tabel.value:.4f})^2 = {e_sq_temp_tabel.value - ex_temp_tabel.value**2:.4f}")

except FileNotFoundError as e:
    print(f"\n[PERINGATAN] Gagal memuat tabel mortalita default: {e}")
    print("Bagian 1 demo (momen berbasis tabel) akan dilewati.")
    # Inisialisasi variabel agar Bagian 3 tidak error jika Bagian 1 gagal
    ex_wl_tabel = None 
    e_sq_wl_tabel = None
    ex_temp_tabel = None
    e_sq_temp_tabel = None

except Exception as e_table:
    print(f"\n[ERROR] Terjadi kesalahan pada perhitungan momen berbasis tabel: {e_table}")
    #traceback.print_exc()
    ex_wl_tabel = None
    e_sq_wl_tabel = None
    ex_temp_tabel = None
    e_sq_temp_tabel = None


# ==============================================================================
# BAGIAN 2: MOMEN CURTATE BERBASIS ASUMSI DISTRIBUSI MURNI
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 2: MOMEN CURTATE BERBASIS ASUMSI DISTRIBUSI MURNI")
print("=" * 70)

assumptions_to_test_moments = [
    ('constant_qx', [qx_konstan_val], f"q_x konstan = {qx_konstan_val}"),
    ('de_moivre', [omega_dm_val], f"De Moivre (ω={int(omega_dm_val)})"),
    ('beta_distribution', [omega_dm_val, alpha_beta_val], f"Beta Dist. (ω={int(omega_dm_val)}, α={alpha_beta_val})"),
    ('gompertz', gompertz_params_val, f"Gompertz (B={gompertz_params_val[0]:.2e}, c={gompertz_params_val[1]})")
]

# Variabel untuk menyimpan salah satu hasil untuk Bagian 3
ex_wl_as_untuk_ops = None # Inisialisasi
e_sq_wl_as_untuk_ops = None # Inisialisasi

for i, (assumption_type, params, desc_short) in enumerate(assumptions_to_test_moments):
    print(f"\n--- 2.{i+1} Menggunakan Asumsi: {desc_short} ---")
    print(f"   Parameter Umum: Usia = {usia_x}, n_temporary = {n_temporary}, Suku Bunga = {suku_bunga_untuk_kalkulator:.2%}")
    print(f"   Parameter Asumsi: {params}\n")

    try:
        # --- Momen Pertama Curtate (e_x dan e_{x:n|}) ---
        ex_wl_as = mp.ex_curtate_assumption(
            age=usia_x, interest_rate=suku_bunga_untuk_kalkulator, 
            assumption_type=assumption_type, params=params # type: ignore
        )
        tampilkan_hasil_momen(f"   a. Ekspektasi Curtate (e_{{{usia_x}}}, Whole Life):", ex_wl_as)

        ex_temp_as = mp.ex_curtate_assumption(
            age=usia_x, interest_rate=suku_bunga_untuk_kalkulator,
            assumption_type=assumption_type, params=params, n_temp=n_temporary # type: ignore
        )
        tampilkan_hasil_momen(f"   b. Ekspektasi Curtate (e_{{{usia_x}:\\overline{{{n_temporary}}}|}}, Temporary):", ex_temp_as)

        # --- Momen Kedua Curtate (E[K_x^2] dan E[(K_{x:n|})^2]) ---
        e_sq_wl_as = mp.e_sq_curtate_assumption(
            age=usia_x, interest_rate=suku_bunga_untuk_kalkulator,
            assumption_type=assumption_type, params=params # type: ignore
        )
        tampilkan_hasil_momen(f"   c. Momen Kedua Curtate (E[K_{{{usia_x}}}^2], Whole Life):", e_sq_wl_as)
        
        e_sq_temp_as = mp.e_sq_curtate_assumption(
            age=usia_x, interest_rate=suku_bunga_untuk_kalkulator,
            assumption_type=assumption_type, params=params, n_temp=n_temporary # type: ignore
        )
        tampilkan_hasil_momen(f"   d. Momen Kedua Curtate (E[K_{{{usia_x}:\\overline{{{n_temporary}}}|}}^2], Temporary):", e_sq_temp_as)

        # --- Variansi Curtate (Var(K_x) dan Var(K_{x:n|})) ---
        var_k_wl_as = mp.var_k_assumption(
            age=usia_x, interest_rate=suku_bunga_untuk_kalkulator,
            assumption_type=assumption_type, params=params # type: ignore
        )
        tampilkan_hasil_momen(f"   e. Variansi Curtate (Var[K_{{{usia_x}}}], Whole Life):", var_k_wl_as,
                              verifikasi_manual_str=f"      -> Verifikasi: {e_sq_wl_as.value:.4f} - ({ex_wl_as.value:.4f})^2 = {e_sq_wl_as.value - ex_wl_as.value**2:.4f}")

        var_k_temp_as = mp.var_k_assumption(
            age=usia_x, interest_rate=suku_bunga_untuk_kalkulator,
            assumption_type=assumption_type, params=params, n_temp=n_temporary # type: ignore
        )
        tampilkan_hasil_momen(f"   f. Variansi Curtate (Var[K_{{{usia_x}:\\overline{{{n_temporary}}}|}}], Temporary):", var_k_temp_as,
                               verifikasi_manual_str=f"      -> Verifikasi: {e_sq_temp_as.value:.4f} - ({ex_temp_as.value:.4f})^2 = {e_sq_temp_as.value - ex_temp_as.value**2:.4f}")
        
        # Simpan salah satu hasil untuk Bagian 3
        if assumption_type == 'de_moivre': # Atau asumsi lain sebagai pembanding
            ex_wl_as_untuk_ops = ex_wl_as # Simpan e_x bukan NSP
            # e_sq_wl_as_untuk_ops = e_sq_wl_as # Jika ingin menggunakan E[K^2] juga

    except Exception as e_assume:
        print(f"   [ERROR] Terjadi kesalahan pada asumsi {assumption_type}: {e_assume}")


# ==============================================================================
# BAGIAN 3: OPERASI ARITMATIKA PADA HASIL `ActuarialResult`
# ==============================================================================
print("\n" + "=" * 70)
print(" BAGIAN 3: OPERASI ARITMATIKA PADA HASIL `ActuarialResult`")
print("=" * 70)

# Pastikan variabel yang dibutuhkan ada sebelum melakukan operasi
ex_wl_as_untuk_ops = locals().get('ex_wl_as_untuk_ops', None) 
ex_wl_tabel_untuk_ops = locals().get('ex_wl_tabel', None) # Ganti nama ini


if ex_wl_tabel_untuk_ops is not None and ex_wl_as_untuk_ops is not None:
    print("\nMenjumlahkan Ekspektasi Hidup dari Tabel dengan Ekspektasi Hidup dari Asumsi De Moivre:")
    total_ex_gabungan = ex_wl_tabel_untuk_ops + ex_wl_as_untuk_ops 
    total_ex_gabungan.show()
else:
    print("\n[PERINGATAN] Contoh operasi aritmatika tidak dapat dijalankan sepenuhnya.")
    print("Pastikan perhitungan momen e_x berbasis tabel dan asumsi De Moivre di atas berhasil.")

print("\n" + "=" * 70)
print("              DEMO LENGKAP MORTAPY SELESAI")
print("=" * 70)

          DEMO LENGKAP PERHITUNGAN MOMEN MORTAPY

--- Parameter Umum yang Digunakan dalam Demo ---
Usia awal (x)         : 65
Suku bunga (i)        : 5.00%
Gender (untuk tabel)  : Wanita
Periode temporary (n) : 10 tahun
--------------------------------------------------

 BAGIAN 1: MOMEN CURTATE BERBASIS TABEL MORTALITA (TMI DEFAULT)

Berhasil memuat tabel default: <MortalityTable path='tabel_mortalita_penduduk_indonesia_2023.csv', max_age=111, gender_specific=True, unisex=False>

--- Menggunakan parameter: Usia = 65, Gender = wanita, n_temporary = 10 ---
1.1.1 Ekspektasi Curtate Future Lifetime (e_{65}, Whole Life, Tabel):


<IPython.core.display.Math object>

Deskripsi: Ekspektasi Curtate Future Lifetime (Tabel), Usia 65, Gender Wanita
--------------------------------------------------
1.1.2 Ekspektasi Curtate Future Lifetime (e_{65:\overline{10}|}, Temporary, Tabel):


<IPython.core.display.Math object>

Deskripsi: Ekspektasi Curtate Future Lifetime 10-tahun temporary (Tabel), Usia 65, Gender Wanita
--------------------------------------------------
1.2.1 Momen Kedua Curtate (E[K_{65}^2], Whole Life, Tabel):


...

Deskripsi: Variansi Curtate Future Lifetime (Asumsi: Gompertz (B=0.0001, c=1.1), Usia 65
      -> Verifikasi: 116.9549 - (9.0266)^2 = 35.4752
--------------------------------------------------
   f. Variansi Curtate (Var[K_{65:\overline{10}|}], Temporary):


<IPython.core.display.Math object>

Deskripsi: Variansi Curtate Future Lifetime 10-tahun temporary (Asumsi: Gompertz (B=0.0001, c=1.1), Usia 65
      -> Verifikasi: 60.4719 - (6.9879)^2 = 11.6406
--------------------------------------------------

 BAGIAN 3: OPERASI ARITMATIKA PADA HASIL `ActuarialResult`

Menjumlahkan Ekspektasi Hidup dari Tabel dengan Ekspektasi Hidup dari Asumsi De Moivre:


<IPython.core.display.Math object>

Deskripsi: Ekspektasi Curtate Future Lifetime (Tabel), Usia 65, Gender Wanita + Ekspektasi Curtate Future Lifetime (Asumsi: De Moivre (ω=100)), Usia 65

              DEMO LENGKAP MORTAPY SELESAI
